# M05 - 02 Acumulados por entidad


In [8]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, min as fmin, sum as fsum, row_number

spark = get_spark("novashop-m05")
orders_gmv = (
    spark.read.parquet(str(STAGING / "fact_lines"))
    .join(spark.read.parquet(str(STAGING / "customers_clean")), "customer_id", "inner")
    .where(col("is_billable"))
    .groupBy("customer_id", "order_id")
    .agg(
        fmin("order_ts").alias("order_ts"),  # un instante por ticket
        fsum("gmv_line").alias("gmv"),       # dinero de todas las líneas del ticket
    )
)
print("pedidos cobrables con cliente", orders_gmv.count())


ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated
pedidos cobrables con cliente 469


## Prueba - Qué pasa si no agrupamos

In [9]:
lineas = (
    spark.read.parquet(str(STAGING / "fact_lines"))
    .join(spark.read.parquet(str(STAGING / "customers_clean")), "customer_id", "inner")
    .where(col("is_billable"))
)
print("líneas", lineas.count(), "pedidos distintos", lineas.select("order_id").distinct().count())


líneas 1122 pedidos distintos 469


## Paso 2 - Número de pedido y acumulado

In [10]:
from pyspark.sql.window import Window

w = Window.partitionBy("customer_id").orderBy("order_ts")
hist = (
    orders_gmv.withColumn("order_n", row_number().over(w))
    .withColumn("gmv_running", fsum("gmv").over(w))
)
hist.orderBy("customer_id", "order_n").show(12)

+-----------+--------+-------------------+------------------+-------+------------------+
|customer_id|order_id|           order_ts|               gmv|order_n|       gmv_running|
+-----------+--------+-------------------+------------------+-------+------------------+
|      C0001|  O00643|2024-06-16 18:00:00|          739.1475|      1|          739.1475|
|      C0001|  O00783|2024-09-26 15:00:00|           652.812|      2|         1391.9595|
|      C0002|  O00161|2024-06-22 11:00:00|            178.36|      1|            178.36|
|      C0002|  O00252|2024-08-17 19:00:00|          2602.781|      2|          2781.141|
|      C0003|  O00350|2024-05-26 17:00:00|             842.3|      1|             842.3|
|      C0003|  O00718|2024-10-10 14:00:00|            186.55|      2|           1028.85|
|      C0005|  O00521|2024-04-01 10:00:00|            869.55|      1|            869.55|
|      C0005|  O00684|2024-08-29 02:00:00|343.14099999999996|      2|1212.6909999999998|
|      C0005|  O00650

## Prueba -- un clinete concreto

In [11]:
# Cambia el id por uno que hayas visto con varios pedidos
cid = hist.where(col("order_n") >= 2).select("customer_id").first()["customer_id"]
print("cliente", cid)
hist.where(col("customer_id") == cid).orderBy("order_n").show()


cliente C0001
+-----------+--------+-------------------+--------+-------+-----------+
|customer_id|order_id|           order_ts|     gmv|order_n|gmv_running|
+-----------+--------+-------------------+--------+-------+-----------+
|      C0001|  O00643|2024-06-16 18:00:00|739.1475|      1|   739.1475|
|      C0001|  O00783|2024-09-26 15:00:00| 652.812|      2|  1391.9595|
+-----------+--------+-------------------+--------+-------+-----------+



## Paso 3 - primera compara vs repetición

In [12]:
hist.groupBy((col("order_n") == 1).alias("is_first")).count().show()

compras_verdad = hist.groupBy((col("order_n") == 1).alias("is_first")).count();
compras_verdad.show()
# compras_fasle = hist.groupBy((col("order_n") == 1).alias("is_first")).count()

+--------+-----+
|is_first|count|
+--------+-----+
|    true|  211|
|   false|  258|
+--------+-----+

+--------+-----+
|is_first|count|
+--------+-----+
|    true|  211|
|   false|  258|
+--------+-----+



## Prueba ventana sobre toda la empresa

In [13]:
w_emp = Window.orderBy("order_ts")  # un solo ranking temporal global
hist_emp = orders_gmv.withColumn("order_n", row_number().over(w_emp))
print("order_n=1 sin partitionBy", hist_emp.where(col("order_n") == 1).count())  # 1
print("order_n=1 con partitionBy", hist.where(col("order_n") == 1).count())     # ~211

26/09/28 16:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 16:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 16:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 16:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 16:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 16:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 1

order_n=1 sin partitionBy 1
order_n=1 con partitionBy 211


## Comprueba


In [14]:
w2 = Window.partitionBy("customer_id").orderBy("order_ts")
crossed = hist.where(col("gmv_running") >= 1000)
first_cross = crossed.withColumn("rn", row_number().over(w2)).where(col("rn") == 1)
first_cross.select("customer_id", "order_n", "gmv_running").show()
print("clientes que cruzan 1000", first_cross.count())

+-----------+-------+------------------+
|customer_id|order_n|       gmv_running|
+-----------+-------+------------------+
|      C0001|      2|         1391.9595|
|      C0002|      2|          2781.141|
|      C0003|      2|           1028.85|
|      C0005|      2|1212.6909999999998|
|      C0007|      1|           1725.21|
|      C0008|      1|          1310.491|
|      C0009|      2|          1077.054|
|      C0012|      2|         1747.6915|
|      C0015|      2|          1810.199|
|      C0016|      3|          2576.876|
|      C0018|      1|           1456.12|
|      C0019|      1|          1391.777|
|      C0020|      1|1311.9219999999998|
|      C0021|      2|          1385.966|
|      C0022|      1|         1529.2525|
|      C0023|      1|          1785.655|
|      C0024|      2|         2434.9045|
|      C0025|      1|1202.9360000000001|
|      C0027|      3|2369.7120000000004|
|      C0028|      1|         1148.8295|
+-----------+-------+------------------+
only showing top